In [1]:
import pandas as pd
from datetime import datetime
import sqlite3
import random
from sqlalchemy import create_engine, text
import os

In [2]:
DB_HOST = os.getenv("DB_HOST", "127.0.0.1")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "vendor_db")
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "postgres")

DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
conn = create_engine(DATABASE_URL)

In [3]:
# conn = sqlite3.connect("./my_database.db")

In [4]:
vendor_ids = pd.read_sql("select distinct(vendor_id) from ticket", con=conn)

In [5]:
test_vendor = vendor_ids.iloc[2]['vendor_id']
test_vendor

'87557121-007b-4e66-9a6e-733b6e2bc363'

In [ ]:
df = pd.read_sql("select * from ticket)

In [6]:
def unassigned_work_orders(vendor_id):
    # df = pd.read_sql(
    #     f"""
    #     select count(*) as unassigned_work_orders
    #     from work_orders
    #     where assigned_vendor='{vendor_id}'"""
    # , con=conn)
    df = pd.read_sql(
        f"""
        select work_order_id, count(*) as "unassigned tickets"
        from ticket t
        join work_orders wo
        using (work_order_id)
        where t.status = 'unassigned' and wo.assigned_vendor = '{vendor_id}'
        group by 1;
        """, con=conn
    )
    return {'vendor_id': vendor_id, 'unassigned tickets': df['unassigned tickets'].sum()}

In [7]:
unassigned_work_orders(test_vendor)

DatabaseError: Execution failed on sql '
        select work_order_id, count(*) as "unassigned tickets"
        from ticket t
        join work_orders wo
        using (work_order_id)
        where t.status = 'unassigned' and wo.assigned_vendor = '87557121-007b-4e66-9a6e-733b6e2bc363'
        group by 1;
        ': (psycopg2.errors.InvalidTextRepresentation) invalid input value for enum core.ticket_status: "unassigned"
LINE 6:         where t.status = 'unassigned' and wo.assigned_vendor...
                                 ^

[SQL: 
        select work_order_id, count(*) as "unassigned tickets"
        from ticket t
        join work_orders wo
        using (work_order_id)
        where t.status = 'unassigned' and wo.assigned_vendor = '87557121-007b-4e66-9a6e-733b6e2bc363'
        group by 1;
        ]
(Background on this error at: https://sqlalche.me/e/20/9h9h)

In [ ]:
def pending_tickets(vendor_id):
    df = pd.read_sql(f"""
    select count(*) as active_tickets
    from ticket 
    where vendor_id='{vendor_id}' and (status = 'in progress' or status = 'assigned')
    """, con=conn)
    count = df['active_tickets'].iloc[0]
    return {'vendor_id': vendor_id, 'active tickets': count}

In [ ]:
pending_tickets(test_vendor)

In [ ]:
df = pd.read_sql(
    f"""select * 
    from ticket 
    where vendor_id='{test_vendor}' and completed_at is not null and created_at > date('now', '-3 months')"""
    , con=conn, parse_dates=['assigned_at', 'completed_at'])

In [ ]:
df['completion_week'] = df['completed_at'].dt.to_period('W').apply(lambda r: r.start_time)

In [ ]:
complete_by_week = df.groupby('completion_week')['ticket_id'].count().reset_index().sort_values(by='ticket_id', ascending=False)
complete_by_week.rename(columns={'ticket_id': 'ticket_count'}, inplace=True)
complete_by_week